# 🏋️ Module 11 — Fine-Tuning Lab: your dataset goes to the gym
**Designed by Engr Usman Hasan** · Bano Qabil Generative AI course

| Step | What happens | Day |
|---|---|---|
| 0 | Turn on the free GPU and install tools | Sat |
| 1 | Load your Module 10 dataset (or the farm demo) | Sat |
| 2 | Load a small model in 4-bit (the "Q" in QLoRA) | Sat |
| 3 | BEFORE: ask the base model some test questions | Sat |
| 4 | Add LoRA and TRAIN | Sat |
| 5 | Loss curves: did it learn? | Sat |
| 6 | AFTER: base vs fine-tuned on the whole test set | Sun |
| 7 | Scoreboard (+ optional Groq / Gemini big model) | Sun |
| 8 | Save your adapter + cost of your training | Sun |

Run cells top to bottom with `Shift + Enter`. Read the comments: every line has one.

## ⚙️ Step 0 — Turn on the free GPU
Menu: **Runtime → Change runtime type → T4 GPU → Save**. Then run the cells below.

In [ ]:
# show the GPU (if this prints an error, you forgot to turn the GPU on)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU found: Runtime > Change runtime type > T4 GPU" 

In [ ]:
# install the fine-tuning tools (exact versions, so the code always works the same)
# transformers = models, peft = LoRA, trl = the trainer, bitsandbytes = 4-bit, datasets = data
!pip install -q "transformers==5.23.0" "peft==0.22.0" "trl==1.18.2" "accelerate==1.16.0" "datasets==5.2.0" "bitsandbytes==0.50.2" groq google-genai
# if Colab asks you to RESTART the session after this, click Restart and run from the next cell

## 📦 Step 1 — Your dataset
**Option A (recommended for your project):** upload the zip you made in Module 10 (with `train.jsonl`, `val.jsonl`, `test.jsonl`).
**Option B:** use the farm demo dataset (order slips) to follow along with the teacher.

In [ ]:
# create the FARM DEMO dataset (150 WhatsApp orders -> JSON slips, from Module 10)
# 'os' makes folders
import os

# file name -> file text
demo = {
    'demo_data/test.jsonl': '{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal subah aadha kilo makkhan rakh dena, main khud le jaunga"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Can you send 1 litres of lassi on Friday? Address: clifton block 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 1, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "parson ke liye 1 kg makhan book kar lein, north nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "abhi ke liye 2 kilo desi ghee book kar lein, north nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"now\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "farm Sunday ko khula hota hai?"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "mujhe makhan chahiye Sunday subah, bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": \\"Sunday morning\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal subah ke liye 2 liter doodh book kar lein, north nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 2, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Clifton blk 5 wala order: aadha kilo makhan, kal shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "abhi dahi bhej dein pechs"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": \\"now\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Is the milk boiled or raw?"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "pechs wala order: aadha kilo ghee, aaj shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Gulshan e Iqbal wala order: 2 kilo dahi, abhi"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"now\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai aaj shaam 10 litre doodh bhej dena DHA ph 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 10, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "mujhe desi ghee chahiye parson, gulshan"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal shaam ke liye aadha kilo makhan book kar lein, pechs"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"PECHS\\"}"}]}\n',
    'demo_data/train.jsonl': '{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "AoA bhai 3 kg desi ghee parson chahiye. address north nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 3, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Friday ko lassi bhej dein clifton block 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "plz 4 ltr doodh dha phase 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 4, \\"unit\\": \\"litre\\", \\"delivery_time\\": null, \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "AoA bhai 1 litre lassi Friday ko chahiye. address DHA ph 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 1, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai rider 1 ghanta late aaya"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "1 liters fresh milk please, deliver this evening to Gulshan e Iqbal"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 1, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Hi, I need 10 litres lassi tomorrow morning at dha phase 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 10, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "clifton block 5 wala order: 1 kg ghee, aaj shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "abhi 3 liter lassi rakh dena, main khud le jaunga"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 3, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"now\\", \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "dahi 1 kg parson tak pechs pohncha dein"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "5 kilos yogurt please, deliver this evening to PECHS"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kya aap paneer bhi bechte hain?"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Can you send half kg of white butter on Friday? Address: clifton block 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "2 litres lassi please, deliver tomorrow morning to bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 2, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "plz 2 kilo makkhan gulshan"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": null, \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai aaj shaam aadha kilo desi ghee bhej dena clifton block 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Thank you for the quick delivery"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Clifton blk 5 wala order: 2 kilo dahi, Friday ko"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai kal shaam 1 kilo makkhan bhej dena Gulshan e Iqbal"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "north nazimabad wala order: 1 kg dahi, parson"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Hi, I need 1 kg pure ghee day after tomorrow at Gulshan e Iqbal"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai aaj shaam 3 kilo dahi bhej dena dha phase 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 3, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "dha phase6 wala order: 3 kilo desi ghee, Friday ko"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 3, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "rider ka number bhej dein"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "clifton block 5 wala order: 10 ltr dudh, abhi"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 10, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"now\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "desi ghee 2 kilo Friday ko tak pechs pohncha dein"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Can you send 4 litres of fresh milk day after tomorrow? Address: pechs"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 4, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "shukriya bhai, order mil gaya"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "parson 2 kilo desi ghee rakh dena, main khud le jaunga"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, dha phase 6 mein 1 liter dudh chahiye Sunday subah"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 1, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"Sunday morning\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Hello, are you open today?"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, gulshan mein 5 kg desi ghee chahiye aaj shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Hi, I need 1 litres fresh milk tomorrow evening at north nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 1, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal shaam aadha kilo desi ghee rakh dena, main khud le jaunga"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "mera address change ho gaya hai"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal wala order cancel kar dein"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Ok"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "plz 1 kilo desi ghee north nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": null, \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "pechs wala order: 1 kilo makkhan, kal shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Bahadurabad wala order: 10 liter lassi, kal shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 10, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, Gulshan e Iqbal mein 2 kilo ghee chahiye Friday ko"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Sunday subah 5 kg makkhan rakh dena, main khud le jaunga"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Sunday morning\\", \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "plz 2 kg ghee N. Nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": null, \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Sunday subah aadha kilo dahi rakh dena, main khud le jaunga"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Sunday morning\\", \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Hi, I need 4 liters milk this evening at PECHS"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 4, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "AoA bhai 3 litre dudh kal subah chahiye. address DHA ph 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 3, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai Friday ko 1 kg desi ghee bhej dena bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "N. Nazimabad wala order: 5 litre dudh, aaj shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 5, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai abhi 5 ltr doodh bhej dena dha phase6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 5, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"now\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "dudh 10 liter parson tak Clifton blk 5 pohncha dein"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 10, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "😊👍"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "doodh 2 liter abhi tak dha phase 6 pohncha dein"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 2, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"now\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai kal shaam 3 kilo makkhan bhej dena clifton block 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 3, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, clifton block 5 mein 3 kilo desi ghee chahiye kal shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 3, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, dha phase 6 mein 3 kg ghee chahiye parson"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 3, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "plz 3 litre lassi PECHS"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 3, \\"unit\\": \\"litre\\", \\"delivery_time\\": null, \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "AoA bhai aadha kilo makkhan Friday ko chahiye. address Bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "doodh ka rate kya hai?"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal subah ke liye 3 kg makkhan book kar lein, pechs"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 3, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "AoA bhai 5 ltr doodh kal subah chahiye. address north nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 5, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Do you deliver in Malir?"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "aaj shaam ke liye 1 kilo ghee book kar lein, Bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "mujhe lassi chahiye parson, Gulshan e Iqbal"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, Clifton blk 5 mein 5 kg makkhan chahiye kal subah"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "AoA bhai 4 liter doodh parson chahiye. address N. Nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 4, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "4 liters fresh milk please, deliver this evening to Gulshan e Iqbal"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 4, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "parson ke liye 2 kilo desi ghee book kar lein, N. Nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal subah ke liye 5 kg ghee book kar lein, Gulshan e Iqbal"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, N. Nazimabad mein 2 kg desi ghee chahiye aaj shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "AoA bhai 2 kilo desi ghee Friday ko chahiye. address DHA ph 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai aaj shaam 5 kg makkhan bhej dena Clifton blk 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai Sunday subah 4 liter lassi bhej dena clifton block 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 4, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"Sunday morning\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal subah ke liye 2 kg desi ghee book kar lein, bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "mujhe doodh chahiye parson, bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "abhi 4 ltr doodh rakh dena, main khud le jaunga"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 4, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"now\\", \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal shaam lassi bhej dein clifton block 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "makkhan 5 kilo parson tak dha phase 6 pohncha dein"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "plz 1 liter doodh gulshan"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 1, \\"unit\\": \\"litre\\", \\"delivery_time\\": null, \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "aaj shaam desi ghee bhej dein pechs"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "ghee pure hai ya mix?"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Clifton blk 5 wala order: 2 kilo makhan, aaj shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai Friday ko 3 liter lassi bhej dena Gulshan e Iqbal"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 3, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Hi, I need 1 liters fresh milk tomorrow evening at N. Nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 1, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai kal subah 10 litre lassi bhej dena gulshan"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 10, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "dha phase 6 wala order: aadha kilo makkhan, parson"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "clifton block 5 wala order: aadha kilo dahi, Friday ko"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "AoA bhai 1 kg makhan Friday ko chahiye. address N. Nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "delivery charges kitne hain DHA mein?"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Please call me back"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Hi, I need 2 kg pure ghee tomorrow morning at bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, bahadurabad mein 10 liter dudh chahiye aaj shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 10, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "1 kg butter please, deliver tomorrow morning to Bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Hi, I need 2 kg dahi tomorrow morning at Bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "plz 3 liter doodh DHA ph 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 3, \\"unit\\": \\"litre\\", \\"delivery_time\\": null, \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, N. Nazimabad mein 1 ltr dudh chahiye Friday ko"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 1, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "lassi chahiye 10 litre Sunday subah"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 10, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"Sunday morning\\", \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, pechs mein 3 kg dahi chahiye parson"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 3, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "10 liter doodh bhejwa dein PECHS please"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 10, \\"unit\\": \\"litre\\", \\"delivery_time\\": null, \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai abhi 2 ltr doodh bhej dena Clifton blk 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 2, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"now\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "aaj ka doodh khatta tha"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "makhan aadha kilo kal subah tak north nazimabad pohncha dein"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, Clifton blk 5 mein 1 kg dahi chahiye aaj shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "AoA bhai 10 litre dudh kal subah chahiye. address N. Nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 10, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Eid Mubarak to the whole Marhaba team!"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai parson 2 litre lassi bhej dena dha phase6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 2, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "aaj shaam 1 kilo dahi rakh dena, main khud le jaunga"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "makkhan 2 kilo kal shaam tak clifton block 5 pohncha dein"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, Gulshan e Iqbal mein 1 kg dahi chahiye Friday ko"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "mujhe lassi chahiye abhi, north nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": \\"now\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, N. Nazimabad mein 1 kilo dahi chahiye abhi"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"now\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "last week ka bill kitna bana?"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "parson ke liye aadha kilo makkhan book kar lein, north nazimabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal shaam aadha kilo dahi rakh dena, main khud le jaunga"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "aadha kilo makkhan bhejwa dein DHA ph 6 please"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": null, \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai parson 1 kilo makhan bhej dena pechs"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "north nazimabad wala order: 1 kilo ghee, parson"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, N. Nazimabad mein aadha kilo dahi chahiye Sunday subah"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Sunday morning\\", \\"area\\": \\"North Nazimabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Can you send 5 kilos of dahi tomorrow morning? Address: PECHS"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai kal shaam 4 litre doodh bhej dena Clifton blk 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 4, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n',
    'demo_data/val.jsonl': '{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "1 kg yogurt please, deliver tomorrow morning to Gulshan e Iqbal"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Can you send half kg of butter this evening? Address: gulshan"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Sunday subah ke liye 1 kilo dahi book kar lein, Clifton blk 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"yogurt\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Sunday morning\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "kal subah aadha kilo desi ghee rakh dena, main khud le jaunga"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 0.5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Can you send 1 kg of butter on Friday? Address: Clifton blk 5"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Clifton Block 5\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Friday ko 1 litre lassi rakh dena, main khud le jaunga"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 1, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": null}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "bhai kal subah 1 kg makhan bhej dena bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"tomorrow morning\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "2 kg butter please, deliver this evening to gulshan"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 2, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"Gulshan-e-Iqbal\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "3 kg makhan bhejwa dein Bahadurabad please"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"butter\\", \\"quantity\\": 3, \\"unit\\": \\"kg\\", \\"delivery_time\\": null, \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, PECHS mein 5 kilo desi ghee chahiye aaj shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 5, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"today evening\\", \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "plz 5 litre lassi PECHS"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"lassi\\", \\"quantity\\": 5, \\"unit\\": \\"litre\\", \\"delivery_time\\": null, \\"area\\": \\"PECHS\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Friday ko ke liye 1 kg desi ghee book kar lein, bahadurabad"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": 1, \\"unit\\": \\"kg\\", \\"delivery_time\\": \\"Friday\\", \\"area\\": \\"Bahadurabad\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "Assalam o alaikum, dha phase6 mein 10 ltr doodh chahiye kal shaam"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"milk\\", \\"quantity\\": 10, \\"unit\\": \\"litre\\", \\"delivery_time\\": \\"tomorrow evening\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "mujhe desi ghee chahiye parson, DHA ph 6"}, {"role": "assistant", "content": "{\\"is_order\\": true, \\"product\\": \\"desi ghee\\", \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": \\"day after tomorrow\\", \\"area\\": \\"DHA Phase 6\\"}"}]}\n{"messages": [{"role": "system", "content": "You turn customer WhatsApp messages for Marhaba Cattle & Dairy Farm into one JSON order slip with the keys is_order, product, quantity, unit, delivery_time and area. Use null for anything the customer did not say. Reply with the JSON only."}, {"role": "user", "content": "JazzCash se payment ho sakti hai?"}, {"role": "assistant", "content": "{\\"is_order\\": false, \\"product\\": null, \\"quantity\\": null, \\"unit\\": null, \\"delivery_time\\": null, \\"area\\": null}"}]}\n',
}
# make the folder
os.makedirs("demo_data", exist_ok=True)
# write each file
for path, text in demo.items():
    # save it
    open(path, "w", encoding="utf-8").write(text)
# confirm
print("Demo data ready:", sorted(demo))

In [ ]:
# ===== OPTION A: upload YOUR Module 10 zip (skip this cell to use the farm demo) =====
# Colab's upload button
from google.colab import files
# 'zipfile' opens zip files
import zipfile
# choose your zip (e.g. my_project.zip)
uploaded = files.upload()
# unzip every uploaded file into the folder "my_data"
for name in uploaded:
    # extract all files
    zipfile.ZipFile(name).extractall("my_data")
# list what we got
print(os.listdir("my_data"))

In [ ]:
# ===== CHOOSE YOUR DATA FOLDER =====
# "demo_data" = farm demo,  "my_data" = your own upload
DATA_DIR = "demo_data"

# 'json' reads .jsonl files
import json
# read a .jsonl file into a list of examples
def read_jsonl(path):
    # one example per non-empty line
    return [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]

# the three parts of the dataset
train_rows = read_jsonl(f"{DATA_DIR}/train.jsonl")
# the practice test
val_rows = read_jsonl(f"{DATA_DIR}/val.jsonl")
# the sealed final exam
test_rows = read_jsonl(f"{DATA_DIR}/test.jsonl")
# sizes
print(f"train: {len(train_rows)}   val: {len(val_rows)}   test: {len(test_rows)}")
# show one training example
print(json.dumps(train_rows[0], indent=2, ensure_ascii=False))

## 🧠 Step 2 — Settings and the base model
| Setting | Simple meaning | Start with |
|---|---|---|
| `MODEL_NAME` | which ready-made model we teach | Qwen2.5 **0.5B** Instruct (fast). Try **1.5B** for better results |
| `EPOCHS` | how many times it reads all examples | 3 |
| `LORA_R` | thickness of the LoRA "notebook" | 16 |
| `LORA_ALPHA` | how loudly the notebook speaks | 2 × r |
| `LEARNING_RATE` | step size of each correction | 0.0002 |

In [ ]:
# ===== SETTINGS (change these and re-run to experiment) =====
# a small "instruct" model that already follows instructions
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# full passes over the training data
EPOCHS = 3
# LoRA rank: size of the small trainable matrices
LORA_R = 16
# LoRA scaling: usually 2 x LORA_R
LORA_ALPHA = 32
# how big each learning step is
LEARNING_RATE = 2e-4
# examples per step (lower it if you get "out of memory")
BATCH_SIZE = 4
# -1 = train for EPOCHS. A positive number = stop after that many steps (quick tests)
MAX_STEPS = -1

In [ ]:
# 'torch' runs the maths on the GPU
import torch
# tools to load a model, its tokenizer, and 4-bit settings
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# do we have a GPU?
GPU = torch.cuda.is_available()
# the tokenizer turns text into token numbers and back
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# pad on the LEFT so we can generate many answers at once
tokenizer.padding_side = "left"

# on a GPU: load the model in 4-bit = QLoRA (4x less memory)
if GPU:
    # the 4-bit recipe from the QLoRA paper (NF4 + float16 maths)
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    # download + load the model, compressed to 4-bit, onto the GPU
    # dtype=float16: T4 GPUs can't do bfloat16 (the model's default), so we ask for float16 on purpose
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quant, device_map="auto", dtype=torch.float16)
# no GPU: normal precision on the CPU (slow, only for quick tests)
else:
    # load in 32-bit on the CPU
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)

# count all the model's numbers (parameters)
total_params = sum(p.numel() for p in model.parameters())
# show size and memory
print(f"Model: {MODEL_NAME}")
# parameters in millions
print(f"Parameters: {total_params/1e6:,.0f} million")
# memory used by the weights
print(f"Memory used: {model.get_memory_footprint()/1e9:.2f} GB  ({'4-bit QLoRA' if GPU else '32-bit, CPU'})")

## 👀 Step 3 — BEFORE training: ask the base model
Same system prompt, same questions. Let's see how it does *without* training.

In [ ]:
# write answers for many conversations at once
def generate(conversations, max_new_tokens=150, batch=8):
    # all answers collected here
    answers = []
    # work through the list in small batches
    for i in range(0, len(conversations), batch):
        # turn each conversation into the model's chat format, ending where the assistant should speak
        texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in conversations[i:i+batch]]
        # text -> token numbers, padded to the same length
        enc = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        # no training here, so switch off gradient tracking (faster)
        with torch.no_grad():
            # greedy decoding (do_sample=False) = the same answer every time
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        # keep only the NEW tokens (the answer), turn them back into text
        answers += tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    # remove spaces around each answer
    return [a.strip() for a in answers]

# the test questions WITHOUT the perfect answer (system + user messages only)
test_prompts = [r["messages"][:-1] for r in test_rows]
# ask the base model the first 3 questions
before = generate(test_prompts[:3])
# show question, perfect answer, base answer
for r, b in zip(test_rows[:3], before):
    # the customer message
    print("INPUT     :", r["messages"][1]["content"])
    # the perfect answer
    print("REFERENCE :", r["messages"][-1]["content"])
    # what the base model said
    print("BASE MODEL:", b, "\n")

## 🏋️ Step 4 — Add LoRA and TRAIN
The big model stays **frozen**. LoRA adds small "notebook" matrices, and **only those** are trained.

In [ ]:
# the datasets library holds the training examples
from datasets import Dataset
# LoRA settings
from peft import LoraConfig
# the trainer for Supervised Fine-Tuning (SFT)
from trl import SFTTrainer, SFTConfig

# turn our rows into prompt (system+user) and completion (assistant) parts
def to_dataset(rows):
    # the model only learns to write the COMPLETION, not to repeat the prompt
    return Dataset.from_list([{"prompt": r["messages"][:-1], "completion": [r["messages"][-1]]} for r in rows])

# LoRA: which layers get a small notebook, and how big it is
lora_config = LoraConfig(
    # rank (thickness of the notebook)
    r=LORA_R,
    # scaling factor
    lora_alpha=LORA_ALPHA,
    # randomly drop 5% of notes while training (prevents memorising)
    lora_dropout=0.05,
    # the attention layers: query, key, value, output
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    # we train a text-generating model
    task_type="CAUSAL_LM",
)

# training settings
train_config = SFTConfig(
    # where checkpoints would go
    output_dir="checkpoints",
    # how many full passes over the data
    num_train_epochs=EPOCHS,
    # quick-test limit (-1 = off)
    max_steps=MAX_STEPS,
    # examples per step
    per_device_train_batch_size=BATCH_SIZE,
    # step size
    learning_rate=LEARNING_RATE,
    # warm up slowly for the first 5 steps
    warmup_steps=5,
    # print the loss every 5 steps
    logging_steps=5,
    # check the practice test (val) after every epoch
    eval_strategy="epoch",
    # T4 GPUs use float16 (they don't support bfloat16)
    fp16=GPU,
    # turn bfloat16 off
    bf16=False,
    # longest example in tokens
    max_length=512,
    # don't save checkpoints (we save the adapter at the end)
    save_strategy="no",
    # don't send logs to online services
    report_to="none",
)

# the trainer puts everything together
trainer = SFTTrainer(model=model, args=train_config, train_dataset=to_dataset(train_rows),
                     eval_dataset=to_dataset(val_rows), processing_class=tokenizer, peft_config=lora_config)
# safety for T4: the small LoRA numbers being trained must be float32 (float16 training needs this)
for p in trainer.model.parameters():
    # only the trainable (LoRA) numbers
    if p.requires_grad:
        # convert them to 32-bit
        p.data = p.data.float()
# how many numbers will actually be trained?
trainer.model.print_trainable_parameters()

In [ ]:
# 🚀 TRAIN! (farm demo on a T4: a few minutes)
result = trainer.train()
# how long training took (needed for the cost calculation on Sunday)
TRAIN_SECONDS = result.metrics["train_runtime"]
# show it
print(f"\nTraining time: {TRAIN_SECONDS/60:.1f} minutes")
# from now on, 'model' means the fine-tuned model
model = trainer.model

## 📉 Step 5 — Did it learn? Read the loss curves
**Loss = mistake score.** Training loss should go **down**. If validation loss goes **up** while training loss goes down → overfitting (memorising).

In [ ]:
# 'matplotlib' draws charts
import matplotlib.pyplot as plt
# the trainer's log: one entry per logging step / evaluation
history = trainer.state.log_history
# training loss points (step, loss)
train_pts = [(h["step"], h["loss"]) for h in history if "loss" in h]
# validation loss points (step, eval_loss)
val_pts = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]
# a new chart
plt.figure(figsize=(8, 4))
# the training line
plt.plot(*zip(*train_pts), marker="o", label="training loss (textbook)")
# the validation line (only if it exists)
if val_pts:
    # draw it
    plt.plot(*zip(*val_pts), marker="s", label="validation loss (practice test)")
# labels
plt.xlabel("training step"); plt.ylabel("loss (lower = better)"); plt.title("Did the model learn?")
# legend and show
plt.legend(); plt.grid(alpha=.3); plt.show()

## 🆚 Step 6 — AFTER: base vs fine-tuned on the WHOLE test set
The sealed exam (`test.jsonl`) is finally opened. Both models get the exact same questions.

In [ ]:
# switch to answering mode (no dropout)
model.eval()
# fine-tuned answers
finetuned_answers = generate(test_prompts)
# turn the LoRA notebook OFF for a moment = the original base model
with model.disable_adapter():
    # base answers
    base_answers = generate(test_prompts)

# save everything into predictions.jsonl (one test question per line)
with open("predictions.jsonl", "w", encoding="utf-8") as f:
    # pair up each question with both answers
    for r, b, ft in zip(test_rows, base_answers, finetuned_answers):
        # input, perfect answer, and the two models
        row = {"input": r["messages"][1]["content"], "reference": r["messages"][-1]["content"], "base": b, "finetuned": ft}
        # one line per question
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
# done
print("Saved predictions.jsonl with", len(test_rows), "questions")

## 🏆 Step 7 — The scoreboard
`evaluate.py` gives every model the same exam (every line commented, open it!).

In [ ]:
%%writefile evaluate.py
"""
evaluate.py  -  Module 11: give every model the SAME exam and print a scoreboard
================================================================================
Designed by Engr Usman Hasan  (Bano Qabil Generative AI course)

WHAT IT DOES
  Reads a predictions file where every line is one test question:

    {"input": "...", "reference": "the perfect answer",
     "base": "what the base model said",
     "finetuned": "what our fine-tuned model said",
     "groq": "what a big prompted model said"}       <- optional extra columns

  Every key except "input" and "reference" is treated as one MODEL.

THE FLOW
  1. read the predictions file
  2. look at the reference answers and decide the task type:
       JSON task (extraction)  ->  valid JSON?  every field right?  each field right?
       label task (classify)   ->  is the label right?
       text task (style)       ->  how many of the right words did it use? (overlap)
  3. score every model the same way
  4. print the scoreboard + a few examples where the models disagree

HOW TO RUN
  python evaluate.py demo_results/farm_predictions.jsonl
  python evaluate.py predictions.jsonl            <- the file your Colab run downloads

No extra packages needed: only Python's built-in modules.
"""

# 'json' reads the predictions file and checks if answers are valid JSON
import json
# 're' (regular expressions) finds a {...} block inside a messy answer
import re
# 'sys' reads the file name typed after the script name
import sys
# Counter counts how often each label appears
from collections import Counter

# these two keys are NOT models
NOT_MODELS = ("input", "reference")


# ---------------------------------------------------------------------
# STEP 1: read the file
# ---------------------------------------------------------------------

# read every line of a .jsonl file into a list of dictionaries
def read_predictions(path):
    # open the file (utf-8 for Urdu letters and emojis)
    with open(path, encoding="utf-8") as f:
        # one dictionary per non-empty line
        return [json.loads(line) for line in f if line.strip()]


# find the model columns, e.g. ["base", "finetuned", "groq"]
def model_names(rows):
    # keep the order in which keys appear in the first row
    names = [k for k in rows[0] if k not in NOT_MODELS]
    # give the list back
    return names


# ---------------------------------------------------------------------
# small helpers that clean up model answers
# ---------------------------------------------------------------------

# models often wrap JSON in ```json ... ``` - take out the {...} part
def extract_json(text):
    # find the first '{' up to the last '}' (re.S lets '.' match new lines)
    match = re.search(r"\{.*\}", text or "", re.S)
    # nothing that looks like JSON
    if not match:
        # signal "not valid"
        return None
    # try to read it as JSON
    try:
        # json.loads turns the text into a Python dictionary
        value = json.loads(match.group(0))
    # broken JSON
    except ValueError:
        # signal "not valid"
        return None
    # only accept a dictionary (an object), not a list or number
    return value if isinstance(value, dict) else None


# make a label easy to compare: "Negative." -> "negative"
def clean_label(text):
    # lowercase and strip spaces
    text = (text or "").strip().lower()
    # keep only the first word (models sometimes add an explanation)
    first = text.split()[0] if text.split() else ""
    # remove punctuation like . , ! " '
    return first.strip(".,!?:;\"'`*")


# split text into a list of lowercase words
def words(text):
    # \w+ = runs of letters/digits
    return re.findall(r"\w+", (text or "").lower())


# ---------------------------------------------------------------------
# STEP 2: decide the task type from the reference answers
# ---------------------------------------------------------------------

# returns "json", "label" or "text"
def detect_task(rows):
    # all the perfect answers
    refs = [r["reference"] for r in rows]
    # how many of them are JSON objects?
    json_count = sum(1 for r in refs if extract_json(r) is not None)
    # more than half are JSON -> extraction task
    if json_count > len(refs) / 2:
        # JSON task
        return "json"
    # short answers (max 3 words) with few different values -> labels
    if all(len(r.split()) <= 3 for r in refs) and len(set(refs)) <= 10:
        # classification task
        return "label"
    # everything else is free text (style / tone tasks)
    return "text"


# ---------------------------------------------------------------------
# STEP 3: the three ways of scoring
# ---------------------------------------------------------------------

# score one model on a JSON task
def score_json(rows, name):
    # counters
    valid = exact = 0
    # field_right[key] = how many times that field was correct
    field_right = Counter()
    # field_total[key] = how many times that field was asked
    field_total = Counter()
    # go through every test question
    for r in rows:
        # the perfect answer as a dictionary
        ref = extract_json(r["reference"])
        # the model's answer as a dictionary (or None)
        pred = extract_json(r.get(name, ""))
        # count valid JSON answers
        if pred is not None:
            # one more valid answer
            valid += 1
        # compare field by field
        all_ok = pred is not None
        # every key in the perfect answer
        for key, value in ref.items():
            # this field was asked once more
            field_total[key] += 1
            # correct only if the model has the same value
            if pred is not None and pred.get(key) == value:
                # one more correct field
                field_right[key] += 1
            # any wrong field means the slip is not fully correct
            else:
                # mark as not fully correct
                all_ok = False
        # all fields correct?
        if all_ok:
            # one more perfect slip
            exact += 1
    # number of questions
    n = len(rows)
    # average field accuracy over all fields
    fields_pct = 100 * sum(field_right.values()) / max(1, sum(field_total.values()))
    # per-field percentages, for the detailed table
    per_field = {k: 100 * field_right[k] / field_total[k] for k in field_total}
    # return everything as a dictionary
    return {"valid JSON %": 100 * valid / n, "all fields right %": 100 * exact / n,
            "field accuracy %": fields_pct, "per_field": per_field}


# score one model on a label task
def score_label(rows, name):
    # count correct labels
    right = sum(1 for r in rows if clean_label(r.get(name)) == clean_label(r["reference"]))
    # mistakes as (should be, model said) pairs, to see what gets confused
    mistakes = Counter((clean_label(r["reference"]), clean_label(r.get(name)))
                       for r in rows if clean_label(r.get(name)) != clean_label(r["reference"]))
    # return the score and the top confusions
    return {"accuracy %": 100 * right / len(rows), "confusions": mistakes.most_common(3)}


# score one model on a free-text task with word overlap (F1)
def score_text(rows, name):
    # list of F1 scores, one per question
    f1s = []
    # go through every question
    for r in rows:
        # words of the perfect answer
        ref = Counter(words(r["reference"]))
        # words of the model's answer
        pred = Counter(words(r.get(name)))
        # words that appear in both (counted fairly)
        common = sum((ref & pred).values())
        # no overlap at all
        if common == 0:
            # F1 is zero
            f1s.append(0.0)
            # next question
            continue
        # precision: how much of the model's answer was right
        precision = common / sum(pred.values())
        # recall: how much of the perfect answer the model covered
        recall = common / sum(ref.values())
        # F1 = balance of precision and recall
        f1s.append(2 * precision * recall / (precision + recall))
    # average over all questions, as a percentage
    return {"word overlap F1 %": 100 * sum(f1s) / len(f1s)}


# ---------------------------------------------------------------------
# STEP 4: run everything and print the scoreboard
# ---------------------------------------------------------------------

# score every model and return {model_name: scores}
def score_all(rows):
    # decide the task type once
    task = detect_task(rows)
    # pick the matching scoring function
    scorer = {"json": score_json, "label": score_label, "text": score_text}[task]
    # score every model with the SAME function (a fair exam)
    return task, {name: scorer(rows, name) for name in model_names(rows)}


# print the results nicely
def print_report(path):
    # read the predictions
    rows = read_predictions(path)
    # score them
    task, results = score_all(rows)
    # header
    print("=" * 64)
    # which file and task
    print(f"SCOREBOARD  ({len(rows)} test questions, task type: {task})")
    # separator
    print("=" * 64)
    # the main metric names (skip the detailed ones)
    metrics = [m for m in next(iter(results.values())) if m not in ("per_field", "confusions")]
    # table header row
    print(f"{'model':<14}" + "".join(f"{m:>22}" for m in metrics))
    # one row per model
    for name, scores in results.items():
        # model name followed by its numbers
        print(f"{name:<14}" + "".join(f"{scores[m]:>21.0f}%" for m in metrics))
    # extra details for JSON tasks
    if task == "json":
        # heading
        print("\nPer field (% correct):")
        # all field names
        fields = list(next(iter(results.values()))["per_field"])
        # header row with model names
        print(f"{'field':<16}" + "".join(f"{n:>12}" for n in results))
        # one row per field
        for field in fields:
            # field name followed by each model's score
            print(f"{field:<16}" + "".join(f"{results[n]['per_field'][field]:>11.0f}%" for n in results))
    # extra details for label tasks
    if task == "label":
        # heading
        print("\nMost common mistakes (should be -> model said):")
        # one line per model
        for name, scores in results.items():
            # print the confusions list
            print(f"  {name}: {scores['confusions']}")
    # show up to 3 questions where base and fine-tuned disagree
    if "base" in results and "finetuned" in results:
        # heading
        print("\nExamples where the two models disagree:")
        # rows with different answers
        diff = [r for r in rows if (r.get("base") or "").strip() != (r.get("finetuned") or "").strip()]
        # print the first three
        for r in diff[:3]:
            # the question
            print(f"\n  INPUT     : {r['input']}")
            # the perfect answer
            print(f"  REFERENCE : {r['reference']}")
            # base model answer (first 160 characters)
            print(f"  BASE      : {(r.get('base') or '')[:160]}")
            # fine-tuned answer
            print(f"  FINETUNED : {(r.get('finetuned') or '')[:160]}")
    # give the numbers back (the notebook uses them for a chart)
    return task, results


# run only when started directly
if __name__ == "__main__":
    # the file must be given
    if len(sys.argv) < 2:
        # show usage and stop
        sys.exit("Usage: python evaluate.py <predictions.jsonl>")
    # run the report
    print_report(sys.argv[1])


In [ ]:
# the scoring functions
from evaluate import print_report
# print the scoreboard and keep the numbers
task, results = print_report("predictions.jsonl")

In [ ]:
# draw the scoreboard as a bar chart
import matplotlib.pyplot as plt
# the main metric for each task type
main_metric = {"json": "all fields right %", "label": "accuracy %", "text": "word overlap F1 %"}[task]
# model names and their scores
names = list(results); scores = [results[n][main_metric] for n in names]
# draw the bars
plt.figure(figsize=(7, 4)); bars = plt.bar(names, scores, color=["#9AA5A8", "#1F9D6B", "#2F6FD6", "#7B5CD6"][:len(names)])
# write the number on each bar
for b, s in zip(bars, scores): plt.text(b.get_x() + b.get_width()/2, s + 1, f"{s:.0f}%", ha="center", fontweight="bold")
# labels and show
plt.ylim(0, 110); plt.ylabel(main_metric); plt.title("Base vs fine-tuned"); plt.show()

### ⭐ Bonus: can a BIG prompted model beat our SMALL fine-tuned one?
Uses your **free** key. In Colab: 🔑 **Secrets** (left sidebar) → add `GROQ_API_KEY` and/or `GEMINI_API_KEY` → turn on *Notebook access*.

In [ ]:
%%writefile api_baseline.py
"""
api_baseline.py  -  Module 11: can a BIG prompted model beat our SMALL fine-tuned one?
======================================================================================
Designed by Engr Usman Hasan  (Bano Qabil Generative AI course)

WHY THIS FILE EXISTS
  Our fine-tuned model is tiny (0.5 billion parameters). Groq and Gemini give us
  free access to much bigger models. This script asks a big model the SAME test
  questions, with the SAME system prompt, and adds its answers as a new column
  in predictions.jsonl. Then evaluate.py can compare all of them fairly.

THE FLOW
  test.jsonl  ->  for each question: system prompt + user text  ->  Groq or Gemini
             ->  answer saved as a new column  ->  predictions.jsonl (updated)

HOW TO RUN  (you need a FREE key in a .env file or in Colab Secrets)
  pip install groq google-genai python-dotenv
  python api_baseline.py demo_data/test.jsonl demo_results/farm_predictions.jsonl --provider groq
  python api_baseline.py demo_data/test.jsonl demo_results/farm_predictions.jsonl --provider gemini
  python evaluate.py demo_results/farm_predictions.jsonl

Model names change over time. If you get "model not found", check:
  Groq:   https://console.groq.com/docs/models
  Gemini: https://ai.google.dev/gemini-api/docs/models
"""

# 'argparse' reads options like --provider groq from the command line
import argparse
# 'json' reads and writes .jsonl files
import json
# 'os' reads API keys from the environment
import os
# 'time' lets us wait between calls (free keys have speed limits)
import time

# try to read keys from a .env file (works without it too)
try:
    # python-dotenv reads .env into the environment
    from dotenv import load_dotenv
    # load GROQ_API_KEY / GEMINI_API_KEY
    load_dotenv()
# python-dotenv not installed
except ImportError:
    # fine: keys can also be set directly in the environment
    pass

# default model for each provider (change if the name has been retired)
DEFAULT_MODELS = {
    # Groq's big free model (70 billion parameters)
    "groq": "llama-3.3-70b-versatile",
    # Gemini alias that always points to the newest Flash model
    "gemini": "gemini-flash-latest",
}


# ---------------------------------------------------------------------
# the two providers
# ---------------------------------------------------------------------

# ask Groq one question and return the answer text
def ask_groq(client, model, system_prompt, user_text):
    # send system + user messages; temperature=0 = most stable answer
    reply = client.chat.completions.create(
        # which model to use
        model=model,
        # the conversation: instruction first, then the user's message
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_text}],
        # no randomness
        temperature=0,
    )
    # take the text out of the reply
    return reply.choices[0].message.content


# ask Gemini one question and return the answer text
def ask_gemini(client, model, system_prompt, user_text):
    # 'types' holds Gemini's settings classes
    from google.genai import types
    # send the user text; the system prompt goes in the config
    reply = client.models.generate_content(
        # which model to use
        model=model,
        # what the user says
        contents=user_text,
        # the instruction + no randomness
        config=types.GenerateContentConfig(system_instruction=system_prompt, temperature=0),
    )
    # the answer text ("" if Gemini returned nothing)
    return reply.text or ""


# build the right client for the chosen provider
def make_client(provider):
    # Groq
    if provider == "groq":
        # read the key
        key = os.getenv("GROQ_API_KEY")
        # stop with a clear message if it is missing
        if not key:
            # explain where to get it
            raise SystemExit("Set GROQ_API_KEY (free key: https://console.groq.com/keys)")
        # import the Groq package
        from groq import Groq
        # create the client and the ask-function
        return Groq(api_key=key), ask_groq
    # Gemini
    key = os.getenv("GEMINI_API_KEY")
    # stop with a clear message if it is missing
    if not key:
        # explain where to get it
        raise SystemExit("Set GEMINI_API_KEY (free key: https://aistudio.google.com/apikey)")
    # import the Google package
    from google import genai
    # create the client and the ask-function
    return genai.Client(api_key=key), ask_gemini


# ---------------------------------------------------------------------
# main
# ---------------------------------------------------------------------

# the main function
def main():
    # describe the command-line options
    parser = argparse.ArgumentParser(description="Add a big prompted model's answers to predictions.jsonl")
    # 1st argument: the test file (has the system prompt and questions)
    parser.add_argument("test_file")
    # 2nd argument: the predictions file to update (created if missing)
    parser.add_argument("predictions_file")
    # which provider
    parser.add_argument("--provider", choices=["groq", "gemini"], default="groq")
    # which model (optional)
    parser.add_argument("--model", default=None)
    # seconds to wait between calls (free keys allow only a few calls per minute)
    parser.add_argument("--wait", type=float, default=2.5)
    # read the options
    args = parser.parse_args()

    # the model to use: the one given, or the default for that provider
    model = args.model or DEFAULT_MODELS[args.provider]
    # column name in predictions.jsonl, e.g. "groq"
    column = args.provider
    # create the client
    client, ask = make_client(args.provider)

    # read the test questions
    with open(args.test_file, encoding="utf-8") as f:
        # one example per line
        tests = [json.loads(line) for line in f if line.strip()]
    # read the existing predictions, if the file exists
    if os.path.exists(args.predictions_file):
        # open it
        with open(args.predictions_file, encoding="utf-8") as f:
            # one row per line
            rows = [json.loads(line) for line in f if line.strip()]
    # no predictions file yet
    else:
        # start with input + reference only
        rows = [{"input": t["messages"][1]["content"], "reference": t["messages"][-1]["content"]} for t in tests]
    # both files must have the same number of questions in the same order
    if len(rows) != len(tests):
        # stop with a clear message
        raise SystemExit("test file and predictions file have a different number of lines")

    # ask every question
    for i, (test, row) in enumerate(zip(tests, rows), start=1):
        # the system prompt (first message)
        system_prompt = test["messages"][0]["content"]
        # the user's message (second message)
        user_text = test["messages"][1]["content"]
        # try up to 3 times (free keys sometimes say "too many requests")
        for attempt in range(3):
            # 'try' so one error doesn't stop everything
            try:
                # ask the model
                row[column] = ask(client, model, system_prompt, user_text).strip()
                # success: leave the retry loop
                break
            # any error (rate limit, network...)
            except Exception as e:
                # show what went wrong
                print(f"  question {i}: {type(e).__name__}, waiting 20s and retrying...")
                # wait before trying again
                time.sleep(20)
        # three failures: store an empty answer so the file stays complete
        else:
            # empty answer counts as wrong in evaluate.py
            row[column] = ""
        # progress line
        print(f"{i}/{len(tests)} done")
        # stay under the free speed limit
        time.sleep(args.wait)

    # save all rows back to the predictions file
    with open(args.predictions_file, "w", encoding="utf-8") as f:
        # one JSON object per line
        for row in rows:
            # ensure_ascii=False keeps Urdu letters readable
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    # tell the student what to do next
    print(f"\nAdded column '{column}' ({model}). Next: python evaluate.py {args.predictions_file}")


# run only when started directly
if __name__ == "__main__":
    # start the program
    main()


In [ ]:
# copy your free keys from Colab Secrets into the environment
try:
    # Colab's secrets helper
    from google.colab import userdata
    # try both keys
    for key_name in ["GROQ_API_KEY", "GEMINI_API_KEY"]:
        # 'try' because a key may not exist
        try:
            # read it into the environment
            os.environ[key_name] = userdata.get(key_name)
            # confirm (never print the key itself!)
            print("Loaded", key_name)
        # missing key
        except Exception:
            # skip it
            pass
# not on Colab
except ImportError:
    # keys can come from a .env file instead
    print("Not on Colab: put keys in a .env file")

In [ ]:
# ask Groq's 70B model the same test questions (adds a 'groq' column), then re-score
if os.environ.get("GROQ_API_KEY"):
    # run the script (about 3 seconds per question because of free-tier limits)
    !python api_baseline.py {DATA_DIR}/test.jsonl predictions.jsonl --provider groq
    # print the new scoreboard
    task, results = print_report("predictions.jsonl")
# no key
else:
    # explain
    print("No GROQ_API_KEY found: add it in Colab Secrets to run this bonus")

In [ ]:
# same with Gemini (adds a 'gemini' column)
if os.environ.get("GEMINI_API_KEY"):
    # run the script
    !python api_baseline.py {DATA_DIR}/test.jsonl predictions.jsonl --provider gemini
    # print the new scoreboard
    task, results = print_report("predictions.jsonl")
# no key
else:
    # explain
    print("No GEMINI_API_KEY found: add it in Colab Secrets to run this bonus")

## 💬 Try it yourself
Type any new message and see what your fine-tuned model says.

In [ ]:
# write your own test message here
MY_MESSAGE = "bhai parson subah 3 litre doodh aur 1 kilo dahi Clifton block 5"
# use the same system prompt as the training data
conversation = [test_rows[0]["messages"][0], {"role": "user", "content": MY_MESSAGE}]
# fine-tuned answer
print("FINE-TUNED:", generate([conversation])[0])
# base answer (LoRA turned off)
with model.disable_adapter():
    # print it
    print("BASE      :", generate([conversation])[0])

## 💾 Step 8 — Save your adapter (your LoRA notebook)

In [ ]:
# save ONLY the small LoRA part (not the whole model)
model.save_pretrained("my_lora_adapter")
# also save the tokenizer settings next to it
tokenizer.save_pretrained("my_lora_adapter")
# measure how big the adapter (the LoRA weights file) is
size_mb = sum(os.path.getsize(os.path.join("my_lora_adapter", f)) for f in os.listdir("my_lora_adapter") if f.startswith("adapter")) / 1e6
# compare with the full model
print(f"Adapter size: {size_mb:.1f} MB   (the full model is about {total_params*2/1e9:.1f} GB in 16-bit)")
# 'shutil' makes zip files
import shutil
# zip the adapter + predictions for download
shutil.make_archive("my_finetune_results", "zip", ".", "my_lora_adapter")
# try to download (works on Colab)
try:
    # Colab's download helper
    from google.colab import files
    # the adapter zip
    files.download("my_finetune_results.zip")
    # the predictions file (for your presentation)
    files.download("predictions.jsonl")
# not on Colab
except ImportError:
    # tell where the files are
    print("Saved my_finetune_results.zip and predictions.jsonl in this folder")

## 💰 What did this cost?
`cost_estimator.py` counts your tokens. Prices are YOUR inputs: look them up on the provider's page.

In [ ]:
%%writefile cost_estimator.py
"""
cost_estimator.py  -  Module 11: what does MY fine-tuning cost?
================================================================
Designed by Engr Usman Hasan  (Bano Qabil Generative AI course)

WHY THIS FILE EXISTS
  Fine-tuning has three bills:
    1. TRAINING on a GPU            (one time)
    2. or TRAINING on a hosted API  (one time, priced per 1 million tokens)
    3. RUNNING the model every day  (every month, forever!)
  Everything depends on TOKENS (small pieces of words), so we count them first.

THE FLOW
  train.jsonl -> count tokens per example -> x epochs = training tokens
             -> put in prices (YOU look them up, they change often) -> the bills

HOW TO RUN
  python cost_estimator.py demo_data
  python cost_estimator.py demo_data --epochs 3 --train-seconds 420
  python cost_estimator.py demo_data --api-price 5 --run-price 0.6 --requests-per-day 500

  --train-seconds  : 'train_runtime' printed at the end of your Colab training
  --gpu-price      : $ per hour for a rented GPU (free Colab = 0)
  --api-price      : $ per 1M training tokens on a hosted fine-tuning service
  --run-price      : $ per 1M tokens to RUN a model (input + output, simple average)
All prices are YOUR inputs. Check the provider's pricing page on the day you calculate.
"""

# 'argparse' reads options like --epochs 3
import argparse
# 'json' reads the .jsonl files
import json
# 'os' builds file paths
import os

# the tokenizer of the model we fine-tune (used if 'transformers' is installed)
TOKENIZER = "Qwen/Qwen2.5-0.5B-Instruct"


# ---------------------------------------------------------------------
# STEP 1: a function that counts tokens
# ---------------------------------------------------------------------

# returns a function: text -> number of tokens
def make_counter():
    # 'try' the real tokenizer first (exact counts)
    try:
        # transformers is installed in Colab
        from transformers import AutoTokenizer
        # download/load the tokenizer (small, a few MB)
        tok = AutoTokenizer.from_pretrained(TOKENIZER)
        # tell the student which method is used
        print(f"Counting tokens with the real tokenizer ({TOKENIZER})")
        # count = length of the token id list
        return lambda text: len(tok(text)["input_ids"])
    # no transformers (or no internet): use a rule of thumb
    except Exception:
        # explain the fallback
        print("transformers not available: estimating 1 token = 4 characters")
        # about 4 characters per token
        return lambda text: len(text) // 4 + 1


# ---------------------------------------------------------------------
# STEP 2: count the tokens of the training file
# ---------------------------------------------------------------------

# returns (number of examples, list of tokens per example)
def count_dataset(path, count):
    # tokens of each example
    sizes = []
    # open the file
    with open(path, encoding="utf-8") as f:
        # one example per line
        for line in f:
            # skip empty lines
            if not line.strip():
                # next line
                continue
            # read the example
            example = json.loads(line)
            # join all message texts (system + user + assistant)
            text = " ".join(m["content"] for m in example["messages"])
            # count its tokens (+10 for the chat-template special tokens)
            sizes.append(count(text) + 10)
    # give back the list
    return sizes


# format dollars nicely: 0.0123 -> "$0.01"
def money(x):
    # two decimals, or "< $0.01" for tiny amounts
    return f"${x:,.2f}" if x >= 0.01 or x == 0 else "< $0.01"


# ---------------------------------------------------------------------
# main
# ---------------------------------------------------------------------

# the main function
def main():
    # describe the options
    p = argparse.ArgumentParser(description="Estimate fine-tuning and running cost")
    # folder with train.jsonl (and test.jsonl)
    p.add_argument("folder")
    # how many times the model reads the whole dataset
    p.add_argument("--epochs", type=int, default=3)
    # measured training time from Colab, in seconds
    p.add_argument("--train-seconds", type=float, default=0)
    # rented GPU price per hour (free Colab = 0)
    p.add_argument("--gpu-price", type=float, default=0)
    # hosted fine-tuning price per 1M training tokens
    p.add_argument("--api-price", type=float, default=0)
    # price per 1M tokens to RUN a model
    p.add_argument("--run-price", type=float, default=0)
    # how many customer messages per day the bot will answer
    p.add_argument("--requests-per-day", type=int, default=1000)
    # read the options
    a = p.parse_args()

    # the token counter
    count = make_counter()
    # count tokens in train.jsonl
    sizes = count_dataset(os.path.join(a.folder, "train.jsonl"), count)
    # tokens in one pass over the data
    per_epoch = sum(sizes)
    # total tokens the model reads during training
    total = per_epoch * a.epochs
    # average size of one example (= roughly one request when running)
    avg = per_epoch / len(sizes)

    # ---- report 1: the dataset
    print("\n1) YOUR DATASET")
    # number of examples
    print(f"   examples in train.jsonl     : {len(sizes)}")
    # average tokens
    print(f"   average tokens per example  : {avg:.0f}")
    # one epoch
    print(f"   tokens in one epoch         : {per_epoch:,}")
    # all epochs
    print(f"   training tokens ({a.epochs} epochs)  : {total:,}")

    # ---- report 2: training on a GPU
    print("\n2) TRAINING ON A GPU (one time)")
    # did the student give the measured time?
    if a.train_seconds:
        # convert seconds to hours
        hours = a.train_seconds / 3600
        # time used
        print(f"   your training time          : {a.train_seconds/60:.1f} minutes")
        # price (free Colab = 0)
        print(f"   cost at {money(a.gpu_price)}/hour        : {money(hours * a.gpu_price)}"
              + ("   (free Colab T4!)" if a.gpu_price == 0 else ""))
    # no time given yet
    else:
        # tell them where to find it
        print("   add --train-seconds <train_runtime from Colab> to calculate")

    # ---- report 3: training on a hosted API
    print("\n3) TRAINING ON A HOSTED FINE-TUNING API (one time)")
    # price given?
    if a.api_price:
        # cost = millions of tokens x price per million
        print(f"   {total/1e6:.3f}M tokens x {money(a.api_price)} per 1M = {money(total/1e6*a.api_price)}")
    # no price given
    else:
        # show the formula
        print(f"   cost = {total/1e6:.3f}M tokens x (price per 1M tokens). Add --api-price")

    # ---- report 4: running the model every month
    print("\n4) RUNNING THE MODEL (every month!)")
    # tokens per month = requests per day x 30 days x tokens per request
    monthly_tokens = a.requests_per_day * 30 * avg
    # print the volume
    print(f"   {a.requests_per_day:,} requests/day x 30 days x {avg:.0f} tokens = {monthly_tokens/1e6:.1f}M tokens/month")
    # price given?
    if a.run_price:
        # monthly bill
        print(f"   at {money(a.run_price)} per 1M tokens = {money(monthly_tokens/1e6*a.run_price)} per month")
    # no price given
    else:
        # show the formula
        print("   add --run-price to calculate the monthly bill")

    # the lesson
    print("\nLESSON: training is paid ONCE. Running is paid EVERY MONTH. "
          "A small fine-tuned model is often cheaper to run than a big prompted one.")


# run only when started directly
if __name__ == "__main__":
    # start the program
    main()


In [ ]:
# your real training time from Step 4, free Colab GPU = $0 per hour
# whole seconds (the command line wants a plain number)
SECONDS = int(TRAIN_SECONDS)
# count tokens and show the three bills
!python cost_estimator.py {DATA_DIR} --epochs {EPOCHS} --train-seconds {SECONDS} --requests-per-day 1000

## ✅ Bring to Sunday's presentation (2 minutes per student)
1. Your task in one line
2. One example: input → base answer → fine-tuned answer
3. Your scoreboard (screenshot of the chart)
4. One thing that went wrong or surprised you
5. Training time and tokens (from the cost cell)

---
*Module 11 designed by Engr Usman Hasan*